In [4]:
import pandas as pd
import re
import datetime
from datetime import datetime, timedelta
import numpy as np
import Weekly_DFS_data_load

#head to view all rows
pd.options.display.max_rows = 1000


#### Data Files Needed to Run
Draftkings, FFA, Vegas (Name & Projection)

In [5]:
merged_df, best_values = Weekly_DFS_data_load.load_weekly_data('2025_12_21')

Original DK rows: 581
Merged rows: 583
Missing projections: 398
Players missing projections:
             Name_Clean Position  Salary
            rashee rice       WR    7400
           alvin kamara       RB    5500
             devin neal       RB    5500
         garrett wilson       WR    5500
         tua tagovailoa       QB    5200
         zonovan knight       RB    4900
          devaughn vele       WR    4700
              joe mixon       RB    4600
        dare ogunbowale       RB    4600
         bhayshul tuten       RB    4500
          dylan sampson       RB    4300
          kenny pickett       QB    4300
         british brooks       RB    4100
          deejay dallas       RB    4000
      teddy bridgewater       QB    4000
            andy dalton       QB    4000
             mike white       QB    4000
         connor bazelak       QB    4000
         trevor etienne       RB    4000
        jonathon brooks       RB    4000
           anthony tyus       RB    4000
     

In [6]:
# Show best values overall (sorted by Points_Per_Dollar)
print("Best values (Points per $1,000):")
cols = ['Name_Clean', 'Position', 'Salary', 'Projection', 'Points_Per_Dollar']
overall = best_values.sort_values('Points_Per_Dollar', ascending=False)[cols].head(10).copy()
# Pretty formatting
overall['Salary'] = overall['Salary'].astype(int).map(lambda x: f"{x:,}")
overall['Projection'] = overall['Projection'].astype(float).map(lambda x: f"{x:.2f}")
overall['Points_Per_Dollar'] = overall['Points_Per_Dollar'].astype(float).map(lambda x: f"{x:.3f}")
print(overall.to_string(index=False))

# Show best values by position including FLEX (RB/WR/TE)
print("\nBest values by position:")
for pos in ['QB', 'RB', 'WR', 'TE', 'DST', 'FLEX']:
    if pos == 'FLEX':
        pos_df = best_values[best_values['Position'].isin(['RB', 'WR', 'TE'])]
    else:
        pos_df = best_values[best_values['Position'] == pos]
    if pos_df.empty:
        print(f"\n{pos}: (no players)")
        continue
    top_pos = pos_df.nlargest(10, 'Points_Per_Dollar')[['Name_Clean', 'Salary', 'Projection', 'Points_Per_Dollar']].copy()
    top_pos['Salary'] = top_pos['Salary'].astype(int).map(lambda x: f"{x:,}")
    top_pos['Projection'] = top_pos['Projection'].astype(float).map(lambda x: f"{x:.2f}")
    top_pos['Points_Per_Dollar'] = top_pos['Points_Per_Dollar'].astype(float).map(lambda x: f"{x:.3f}")
    print(f"\n{pos}:")
    print(top_pos.to_string(index=False))

Best values (Points per $1,000):
     Name_Clean Position Salary Projection Points_Per_Dollar
jacoby brissett       QB  5,300      18.24             3.442
 justin herbert       QB  6,000      19.36             3.227
      cj stroud       QB  5,500      17.42             3.167
   tyler shough       QB  5,400      16.78             3.107
    jj mccarthy       QB  4,900      15.02             3.065
         bo nix       QB  6,100      18.26             2.993
   kirk cousins       QB  4,800      14.27             2.973
     josh allen       QB  7,200      21.30             2.958
     josh allen       QB  7,200      21.30             2.958
 baker mayfield       QB  5,700      16.86             2.958

Best values by position:

QB:
     Name_Clean Salary Projection Points_Per_Dollar
jacoby brissett  5,300      18.24             3.442
 justin herbert  6,000      19.36             3.227
      cj stroud  5,500      17.42             3.167
   tyler shough  5,400      16.78             3.107
    j

## Old lineup creation of player combinations

Using for loops is very computational intensive. Working on other solutions to speed up lineup creation

## New lineup creation using numpy

In [7]:
# ...existing code...
import numpy as np
from itertools import combinations
import math

# If we want to see a lot of print statements
VERBOSE = True

# Filter out players without projections and sort by value
valid_players = merged_df[merged_df['Projection'].notna()].copy()

# Get top players by position (expand pools since we need more)
qbs = valid_players[valid_players['Position'] == 'QB'].nlargest(10, 'Points_Per_Dollar')
rbs = valid_players[valid_players['Position'] == 'RB'].nlargest(20, 'Points_Per_Dollar')
wrs = valid_players[valid_players['Position'] == 'WR'].nlargest(20, 'Points_Per_Dollar')
tes = valid_players[valid_players['Position'] == 'TE'].nlargest(10, 'Points_Per_Dollar')
dsts = valid_players[valid_players['Position'] == 'DST'].nlargest(10, 'Points_Per_Dollar')

#Total combinations
total_combos = (len(qbs) * math.comb(len(rbs), 2) *
                math.comb(len(wrs), 3) * len(tes) * len(dsts) *
                (len(rbs) - 2 + len(wrs) - 3 + len(tes)))  # FLEX options

print(f"Total possible lineups with expanded pools: {total_combos}")

max_salary = 50000

# Convert pools to NumPy
def arrs(df):
    return (
        df["Salary"].to_numpy(np.int32),
        df["Projection"].to_numpy(np.float32),
        df["Name_Clean"].to_numpy(object),
    )

# Pools as arrays
qb_sal, qb_pts, qb_names   = arrs(qbs) #Creates three seperate 1D arrays. 1 each for qb salary, points, and names. Arrays are numbers/indexes
rb_sal, rb_pts, rb_names   = arrs(rbs)
wr_sal, wr_pts, wr_names   = arrs(wrs)
te_sal, te_pts, te_names   = arrs(tes)
dst_sal, dst_pts, dst_names= arrs(dsts)

# Number of players per position
n_qb, n_rb, n_wr, n_te, n_dst = len(qb_sal), len(rb_sal), len(wr_sal), len(te_sal), len(dst_sal)

if VERBOSE:
    from pprint import pprint

    def show_pool(tag, sal, pts, names, k=5):
        print(f"[{tag}] n={len(sal)} | salary dtype={sal.dtype}, range=({sal.min()}-{sal.max()}) "
        f"| pts dtype={pts.dtype}, range=({pts.min():.2f}-{pts.max():.2f})")
        print("    sample names:", end=" "); pprint(list(names[:k]))

    print("Pool sizes (QB, RB, WR, TE, DST):", (n_qb, n_rb, n_wr, n_te, n_dst))
    show_pool("QB",  qb_sal, qb_pts, qb_names)
    show_pool("RB",  rb_sal, rb_pts, rb_names)
    show_pool("WR",  wr_sal, wr_pts, wr_names)
    show_pool("TE",  te_sal, te_pts, te_names)
    show_pool("DST", dst_sal, dst_pts, dst_names)

    base_total = n_qb * (n_rb*(n_rb-1)//2) * (n_wr*(n_wr-1)*(n_wr-2)//6) * n_te * n_dst
    print(f"Estimated base combos (Q × C(RB,2) × C(WR,3) × TE × DST): {base_total:,}")
# ...existing code...

# Precompute RB pairs and WR trios
# Because we need to select 2 RBs and 3 WRs, we can precompute all combinations(not permuatations)
# np.arange (like python range) creates a 1-dim array of integers from 0 to n_rb-1
# combinations from itertools generates all possible combinations of the specified length
# list converts the combinations generator to a list
# np.array converts the list to a numpy array of specified dtype
# np.arange creates an array of indicies, cominations creates all combos in tuples, 
# / list convertes to a list, and np.array converts back to an array
rb_pairs = np.array(list(combinations(np.arange(n_rb, dtype=np.int16), 2)), dtype=np.int16)  # (Crb,2)
wr_trios = np.array(list(combinations(np.arange(n_wr, dtype=np.int16), 3)), dtype=np.int16) # (Cwr,3)

Crb, Cwr = len(rb_pairs), len(wr_trios)

# Look up combined salaries and points for each RB pair and WR trio
# Still an array. Takes the indicies from rb_pairs, and matches the indicies to rb_sal, 
# then sums the salaries for each pair. Axis=1 means sum across the columns for each row / 1d array.
rb_pair_sal = rb_sal[rb_pairs].sum(axis=1).astype(np.int32)
rb_pair_pts = rb_pts[rb_pairs].sum(axis=1).astype(np.float32)

wr_trio_sal = wr_sal[wr_trios].sum(axis=1).astype(np.int32)
wr_trio_pts = wr_pts[wr_trios].sum(axis=1).astype(np.float32)


# ...existing code...
if VERBOSE:
    print(f"RB pairs: {len(rb_pairs):,}, WR trios: {len(wr_trios):,}")
    print("First 3 RB pairs (idx):", rb_pairs[:3].tolist())
    print("First 2 WR trios (idx):", wr_trios[:2].tolist())
    print("First 3 RB pair names:", [(rb_names[i], rb_names[j]) for i,j in rb_pairs[:3]])
    print("First 2 WR trio names:", [(wr_names[i], wr_names[j], wr_names[k]) for i,j,k in wr_trios[:2]])
    print("First 3 RB pair salaries:", rb_pair_sal[:3].tolist())
    print("First 2 WR trio salaries:", wr_trio_sal[:2].tolist())
    print("First 3 RB pair points:", rb_pair_pts[:3].tolist())
    print("First 2 WR trio points:", wr_trio_pts[:2].tolist())
# 


# Build FLEX pool (RB + WR + TE) so 20 20 10 would be 50 players
# Concat joins the three 1-d arrays into a single 1d array of int32
# The data is laid out like this:
# [RB0, RB1, ..., RB19, WR0, WR1, ..., WR19, TE0, TE1, ..., TE9]
flex_salary = np.concatenate([rb_sal, wr_sal, te_sal]).astype(np.int32) #joins the three arrays into a single 1d array of int32
flex_points = np.concatenate([rb_pts, wr_pts, te_pts]).astype(np.float32)
flex_names  = np.concatenate([rb_names, wr_names, te_names])

# number of FLEX candidates
n_flex = len(flex_salary)

# Is the flex position an RB, WR, or TE?
# FLEX position/type flags and local indices
# Start with np.zeros and set True for each position
# The n_flex is built from flex_salary which is concatenated from RB, WR, TE, so we can slice accordingly
flex_is_rb = np.zeros(n_flex, dtype=bool); flex_is_rb[:n_rb] = True
flex_is_wr = np.zeros(n_flex, dtype=bool); flex_is_wr[n_rb:n_rb+n_wr] = True
flex_is_te = np.zeros(n_flex, dtype=bool); flex_is_te[n_rb+n_wr:] = True

# We do this beacuse we need to know which player in the FLEX pool corresponds to which RB/WR/TE index 
# np.Full params are shape, fill_vlaue. Fill th idx arrays with n_flex -1s 
# Then for each position, we fill in the indices where that position exists in the FLEX pool

flex_rb_idx = np.full(n_flex, -1, dtype=np.int16); flex_rb_idx[:n_rb] = np.arange(n_rb, dtype=np.int16)
flex_wr_idx = np.full(n_flex, -1, dtype=np.int16); flex_wr_idx[n_rb:n_rb+n_wr] = np.arange(n_wr, dtype=np.int16)
flex_te_idx = np.full(n_flex, -1, dtype=np.int16); flex_te_idx[n_rb+n_wr:] = np.arange(n_te, dtype=np.int16)

if VERBOSE:
    print("flex_is_rb:", flex_is_rb) # [True True True True True True True True True True False False False False False False False False False False False False False False False]
    print("flex_rb_idx:", flex_rb_idx)  # [ 0  1  2  3  4  5  6  7  8  9 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]    
    #print("flex_wr_idx:", flex_wr_idx)  # [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1  0  1  2  3  4  5  6  7  8  9 -1 -1 -1 -1 -1]
    #print("flex_te_idx:", flex_te_idx)  # [-1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1  0  1  2  3  4]
    
    # Example: we already chose RB indices (0,1). Which FLEX slots are safe for RB duplication?
    chosen_rb_pair = np.array([0, 1], dtype=np.int16)
    # np.isin params are element, test_elements
    is_duplicate_rb = np.isin(flex_rb_idx, chosen_rb_pair)  # True where FLEX is RB0 or RB1
    print("dup RB flags:", is_duplicate_rb)  # [ True  True False False False False]
    


# Precompute FLEX overlap masks versus RB-pairs / WR-trios / TE
# Shapes: (Crb, n_flex) 190x50, (Cwr, n_flex)1140x50, (n_te, n_flex) 10x50
# Shapes are 1row for each pair, and a column for each FLEX candidate
# As long as they don't overlap, FLEX candidate is OK
# rb_flex_ok equals True where FLEX can be that position without duplication of already chosen RBs/WRs/TE

# this "(~flex_is_rb)[None, :]" means flex is ok if flex is wr or te (not rb)
# or if flex rb idx is not equal to either of the chosen rb pair
# Flex_is_rb is a 1d array of bools, where the RB indicies are True, 
# ~ is the not operator, which inverts the True/False values
# [None, :] adds a new axis at the front, converting it to a 2d array with 1 row and n_flex columns
# Cant compare the arrays without adding the new axis 
# list is shaped (n,) and flex_rb_idx is shaped (n_flex,) [True True... False False...]
# (~flex_is_rb)[None, :] results in a (1, n_flex) array  [[True True... False False...]]
# Flex_rb is ok for 

# Can we choose this RB for FLEX without duplicating already chosen RBs?
rb_flex_ok = (~flex_is_rb)[None, :] | (
    (flex_rb_idx[None, :] != rb_pairs[:, None, 0]) &
    (flex_rb_idx[None, :] != rb_pairs[:, None, 1])
)

wr_flex_ok = (~flex_is_wr)[None, :] | (
    (flex_wr_idx[None, :] != wr_trios[:, None, 0]) &
    (flex_wr_idx[None, :] != wr_trios[:, None, 1]) &
    (flex_wr_idx[None, :] != wr_trios[:, None, 2])
)

te_flex_ok = (~flex_is_te)[None, :] | (
    (flex_te_idx[None, :] != np.arange(n_te, dtype=np.int16)[:, None])
)

# The shape shows how many RB pairs, WR trios, and TEs we have versus the FLEX pool size
# THe flex pool size is 25 here because we have 10 RBs + 10 WRs + 5 TEs
if VERBOSE:
    print("RB-FLEX overlap mask shape:", rb_flex_ok.shape)
    print("WR-FLEX overlap mask shape:", wr_flex_ok.shape)
    print("TE-FLEX overlap mask shape:", te_flex_ok.shape)

# Goal here to generate all possible base lineups without FLEX
# Then we will add FLEX on top of these base lineups
# Build base grid (QB × RBpair × WRtrio × TE × DST)
# meshgrid creates a rectangular grid from two or more 1-D arrays
# we use this grid because we need to evaluate all combinations of these positions
# This is a 5D grid with one axis per position group
# This like a 5 nested for loop over all positions. For qb in range(n_qb): for rbp in range(Crb): etc, 
iq, irp, iwt, ite, idst = np.meshgrid(
    np.arange(n_qb, dtype=np.int32), # QBs
    np.arange(Crb, dtype=np.int32), # RB pairs
    np.arange(Cwr, dtype=np.int32), # WR trios
    np.arange(n_te, dtype=np.int32),
    np.arange(n_dst, dtype=np.int32),
    indexing='ij'
)

# Flatten base indices
# Each of these will be a 1D array where each index corresponds to a unique combination of (QB, RB pair, WR trio, TE, DST)
iq = iq.ravel() 
irp = irp.ravel() 
iwt = iwt.ravel() 
ite = ite.ravel() 
idst = idst.ravel()

#iq   = [QB0, QB0, QB0, QB0, ... QB1, QB1, ...]
#irp  = [RBpair0, RBpair0, RBpair0, RBpair0, ... RBpair1, ...]
#iwt  = [WRtrio0, WRtrio1, WRtrio2, ...]

# Compute base salaries and points
base_salary = (
    qb_sal[iq] + rb_pair_sal[irp] + wr_trio_sal[iwt] + te_sal[ite] + dst_sal[idst]
).astype(np.int32)

# Compute base points
base_points = (
    qb_pts[iq] + rb_pair_pts[irp] + wr_trio_pts[iwt] + te_pts[ite] + dst_pts[idst]
).astype(np.float32)

# Prune bases that cannot fit even the cheapest FLEX
min_flex = int(flex_salary.min())
can_fit_any_flex = base_salary <= (max_salary - min_flex) # base salary must leave room for cheapest FLEX

# Apply pruning
# 
iq = iq[can_fit_any_flex]; irp = irp[can_fit_any_flex]; iwt = iwt[can_fit_any_flex]
ite = ite[can_fit_any_flex]; idst = idst[can_fit_any_flex]
base_salary = base_salary[can_fit_any_flex]; base_points = base_points[can_fit_any_flex]

print(f"Base combos after pruning: {len(base_salary):,}")


Total possible lineups with expanded pools: 9747000000
Pool sizes (QB, RB, WR, TE, DST): (10, 20, 20, 10, 10)
[QB] n=10 | salary dtype=int32, range=(4800-7200) | pts dtype=float32, range=(14.27-21.30)
    sample names: ['jacoby brissett',
 'justin herbert',
 'cj stroud',
 'tyler shough',
 'jj mccarthy']
[RB] n=20 | salary dtype=int32, range=(5000-8800) | pts dtype=float32, range=(10.50-24.65)
    sample names: ['jahmyr gibbs',
 'bijan robinson',
 'kenneth gainwell',
 'devon achane',
 'javonte williams']
[WR] n=20 | salary dtype=int32, range=(3700-8600) | pts dtype=float32, range=(8.10-21.09)
    sample names: ['dk metcalf',
 'jamarr chase',
 'amonra st brown',
 'jakobi meyers',
 'courtland sutton']
[TE] n=10 | salary dtype=int32, range=(2900-7600) | pts dtype=float32, range=(7.10-18.31)
    sample names: ['harold fannin',
 'travis kelce',
 'dalton schultz',
 'juwan johnson',
 'kyle pitts']
[DST] n=10 | salary dtype=int32, range=(2000-2900) | pts dtype=float32, range=(3.77-5.65)
    sam

In [8]:
# Enumerate FLEX with masks in chunks, keep only top_k to limit memory
top_k = 1_000_000  # top  k lineups based on points but not including FLEX yet
keep_idx = np.empty((0,), dtype=np.int64) # np.empty array to hold indices of kept lineups 
keep_iflex = np.empty((0,), dtype=np.int32) # np.empty array to hold indices of kept FLEX players
keep_points = np.empty((0,), dtype=np.float32)
keep_salary = np.empty((0,), dtype=np.int32)

# Process in chunks to limit memory usage
chunk = 20000
for start in range(0, len(base_salary), chunk): # iterate over base lineups in chunks which don't include the flex position yet
    end = min(start + chunk, len(base_salary))
    bs = base_salary[start:end]
    bp = base_points[start:end]
    _irp = irp[start:end]; _iwt = iwt[start:end]; _ite = ite[start:end]

    # Allowed FLEX by overlap
    allowed = rb_flex_ok[_irp] & wr_flex_ok[_iwt] & te_flex_ok[_ite]  # (chunk, n_flex)
    # Salary feasibility
    rem = (max_salary - bs)[:, None]
    allowed &= (flex_salary[None, :] <= rem)

    rows, cols = np.where(allowed)
    if rows.size == 0:
        continue

    pts = bp[rows] + flex_points[cols]
    sal = bs[rows] + flex_salary[cols]
    idx = (start + rows).astype(np.int64)

    # Append and retain top_k by points
    keep_idx = np.concatenate([keep_idx, idx], axis=0)
    keep_iflex = np.concatenate([keep_iflex, cols.astype(np.int32)], axis=0)
    keep_points = np.concatenate([keep_points, pts], axis=0)
    keep_salary = np.concatenate([keep_salary, sal], axis=0)

    # if keep_points.size > top_k * 2:
    #     kth = np.argpartition(keep_points, -top_k)[-top_k:]
    #     keep_idx, keep_iflex, keep_points, keep_salary = (
    #         keep_idx[kth], keep_iflex[kth], keep_points[kth], keep_salary[kth]
    #     )

#Final top_k trim
# if keep_points.size > top_k:
#     kth = np.argpartition(keep_points, -top_k)[-top_k:]
#     keep_idx, keep_iflex, keep_points, keep_salary = (
#         keep_idx[kth], keep_iflex[kth], keep_points[kth], keep_salary[kth]
#     )
    TRIM_THRESHOLD = top_k * 1.1  # keep buffer small

    if keep_points.size > TRIM_THRESHOLD:
        # Add slight noise to preserve diversity
        noise = np.random.normal(0, 0.1, keep_points.size)
        rank = keep_points + noise

        kth = np.argpartition(rank, -top_k)[-top_k:]
        keep_idx, keep_iflex, keep_points, keep_salary = (
            keep_idx[kth], keep_iflex[kth], keep_points[kth], keep_salary[kth]
        )

# Expand indices back to components for the kept rows
sel_iq  = iq[keep_idx]
sel_irp = irp[keep_idx]
sel_iwt = iwt[keep_idx]
sel_ite = ite[keep_idx]
sel_id  = idst[keep_idx]
sel_ifx = keep_iflex

# Unpack RB pair and WR trio members
rb1 = rb_pairs[sel_irp][:, 0]; rb2 = rb_pairs[sel_irp][:, 1]
wr1 = wr_trios[sel_iwt][:, 0]; wr2 = wr_trios[sel_iwt][:, 1]; wr3 = wr_trios[sel_iwt][:, 2]

# Build a lightweight lineup table (indices + totals). Convert to dicts only if needed later.
lineups_np = {
    "points": keep_points,
    "salary": keep_salary,
    "QB": sel_iq, "RB1": rb1, "RB2": rb2, "WR1": wr1, "WR2": wr2, "WR3": wr3, "TE": sel_ite, "DST": sel_id, "FLEX": sel_ifx
}

# Optional: materialize top N as dicts with names for downstream exposure logic
# 
#N_materialize = 40000
order = np.argsort(-lineups_np["points"])


def flex_name(idx):
    if idx < n_rb:
        return rb_names[idx], rb_sal[idx], rb_pts[idx], "RB"
    elif idx < n_rb + n_wr:
        j = idx - n_rb
        return wr_names[j], wr_sal[j], wr_pts[j], "WR"
    else:
        j = idx - n_rb - n_wr
        return te_names[j], te_sal[j], te_pts[j], "TE"
    
# Exposure-aware direct selection (replaces bulk materialization)
target_N = 10                  # number of lineups desired
max_exposure_pct = 0.3        # per-player cap (e.g. 40%)
max_per_player = max(1, math.floor(target_N * max_exposure_pct))

from collections import defaultdict
usage = defaultdict(int)
selected = []

for k in order:   # order already sorted by descending points
    if len(selected) >= target_N:
        break

    qb_i, rb1_i, rb2_i, wr1_i, wr2_i, wr3_i, te_i, dst_i, fx_i = (
        lineups_np["QB"][k], lineups_np["RB1"][k], lineups_np["RB2"][k],
        lineups_np["WR1"][k], lineups_np["WR2"][k], lineups_np["WR3"][k],
        lineups_np["TE"][k], lineups_np["DST"][k], lineups_np["FLEX"][k]
    )
    fx_name = flex_name(fx_i)  # (name, salary, points, position)

    candidate_players = [
        qb_names[qb_i],
        rb_names[rb1_i], rb_names[rb2_i],
        wr_names[wr1_i], wr_names[wr2_i], wr_names[wr3_i],
        te_names[te_i],
        fx_name[0],
        dst_names[dst_i],
    ]

    if any(usage[p] >= max_per_player for p in candidate_players):
        continue

    lineup = {
        "points": float(lineups_np["points"][k]),
        "salary": int(lineups_np["salary"][k]),
        "QB":  (qb_names[qb_i],  int(qb_sal[qb_i]),  float(qb_pts[qb_i])),
        "RB1": (rb_names[rb1_i], int(rb_sal[rb1_i]), float(rb_pts[rb1_i])),
        "RB2": (rb_names[rb2_i], int(rb_sal[rb2_i]), float(rb_pts[rb2_i])),
        "WR1": (wr_names[wr1_i], int(wr_sal[wr1_i]), float(wr_pts[wr1_i])),
        "WR2": (wr_names[wr2_i], int(wr_sal[wr2_i]), float(wr_pts[wr2_i])),
        "WR3": (wr_names[wr3_i], int(wr_sal[wr3_i]), float(wr_pts[wr3_i])),
        "TE":  (te_names[te_i],  int(te_sal[te_i]),  float(te_pts[te_i])),
        "FLEX": fx_name,
        "DST": (dst_names[dst_i], int(dst_sal[dst_i]), float(dst_pts[dst_i])),
    }
    selected.append(lineup)
    for p in candidate_players:
        usage[p] += 1

print(f"Selected {len(selected)} exposure-capped lineups (cap {max_exposure_pct:.0%}, max_per_player={max_per_player})")

for i, lu in enumerate(selected, 1):
    print(f"\nLineup {i}: points={lu['points']:.2f}, salary={lu['salary']}")
    print(f"  QB:   {lu['QB'][0]} ({lu['QB'][1]}), {lu['QB'][2]:.2f} pts")
    print(f"  RB1:  {lu['RB1'][0]} ({lu['RB1'][1]}), {lu['RB1'][2]:.2f} pts")
    print(f"  RB2:  {lu['RB2'][0]} ({lu['RB2'][1]}), {lu['RB2'][2]:.2f} pts")
    print(f"  WR1:  {lu['WR1'][0]} ({lu['WR1'][1]}), {lu['WR1'][2]:.2f} pts")
    print(f"  WR2:  {lu['WR2'][0]} ({lu['WR2'][1]}), {lu['WR2'][2]:.2f} pts")
    print(f"  WR3:  {lu['WR3'][0]} ({lu['WR3'][1]}), {lu['WR3'][2]:.2f} pts")
    print(f"  TE:   {lu['TE'][0]}  ({lu['TE'][1]}),  {lu['TE'][2]:.2f} pts")
    fxn, fxsal, fxpts, fxpos = lu['FLEX']
    print(f"  FLEX: {fxn} [{fxpos}] ({fxsal}), {fxpts:.2f} pts")
    print(f"  DST:  {lu['DST'][0]} ({lu['DST'][1]}), {lu['DST'][2]:.2f} pts")


Selected 3 exposure-capped lineups (cap 30%, max_per_player=3)

Lineup 1: points=135.89, salary=50000
  QB:   jacoby brissett (5300), 18.24 pts
  RB1:  jahmyr gibbs (8500), 24.65 pts
  RB2:  bijan robinson (8700), 24.07 pts
  WR1:  dk metcalf (5400), 13.61 pts
  WR2:  jakobi meyers (5500), 13.12 pts
  WR3:  courtland sutton (5700), 13.50 pts
  TE:   dalton schultz  (4200),  11.28 pts
  FLEX: harold fannin [TE] (4500), 12.53 pts
  DST:  browns (2200), 4.89 pts

Lineup 2: points=135.89, salary=50000
  QB:   jacoby brissett (5300), 18.24 pts
  RB1:  jahmyr gibbs (8500), 24.65 pts
  RB2:  bijan robinson (8700), 24.07 pts
  WR1:  dk metcalf (5400), 13.61 pts
  WR2:  jakobi meyers (5500), 13.12 pts
  WR3:  courtland sutton (5700), 13.50 pts
  TE:   harold fannin  (4500),  12.53 pts
  FLEX: dalton schultz [TE] (4200), 11.28 pts
  DST:  browns (2200), 4.89 pts

Lineup 3: points=135.70, salary=49900
  QB:   jacoby brissett (5300), 18.24 pts
  RB1:  jahmyr gibbs (8500), 24.65 pts
  RB2:  bijan r